# MalariAI — Phase 5: YOLOv8 SOTA Baseline (Kaggle)

Trains YOLOv8 on the **same BBBC041 dataset** used for Baseline A (Faster R-CNN), reproducing the identical seed-42 966/242 train/val split, so the two baselines are directly comparable in the CBM resubmission.

**Before running:** Settings (right sidebar) -> Accelerator = GPU (T4 x2 or P100) and Internet = On (needed for `pip install ultralytics` and the YOLOv8 pretrained-weight download). Add the `bbbc041-malaria` dataset via + Add Input if it isn't already attached.

In [ ]:
!pip install -q ultralytics

## 1. Locate the dataset
Auto-detects `training.json` under `/kaggle/input/` so it works regardless of the exact mount path.

In [ ]:
import glob, os
from pathlib import Path

matches = glob.glob('/kaggle/input/**/training.json', recursive=True)
assert matches, 'training.json not found under /kaggle/input -- attach the bbbc041-malaria dataset via + Add Input'
DATA_ROOT = Path(matches[0]).parent
TRAIN_JSON = DATA_ROOT / 'training.json'
TEST_JSON  = DATA_ROOT / 'test.json'
IMG_DIR    = DATA_ROOT / 'images'

OUT_DIR = Path('/kaggle/working/yolo_data')
CKPT_DIR = Path('/kaggle/working/checkpoints-kaggle-80epoch')

print('DATA_ROOT :', DATA_ROOT)
print('TRAIN_JSON:', TRAIN_JSON, TRAIN_JSON.exists())
print('TEST_JSON :', TEST_JSON, TEST_JSON.exists())
print('IMG_DIR   :', IMG_DIR, IMG_DIR.exists())

## 2. Label map (same 7 classes as Baseline A -- index 0 = background, unused by YOLO)

In [ ]:
LABEL_TO_INT = {
    'background':     0,
    'red blood cell': 1,
    'trophozoite':    2,
    'ring':           3,
    'schizont':       4,
    'gametocyte':     5,
    'leukocyte':      6,
}
FOREGROUND_NAMES = ['red blood cell','trophozoite','ring','schizont','gametocyte','leukocyte']
SKIP_LABELS = {'difficult'}

## 3. MalariaDataset (mirrors Phase1-EDA/dataset.py) + identical seed-42 80/20 split
Same class, same `torch.utils.data.random_split(seed=42)` call as `Phase2-BaselineA/train_frcnn.py` -- guarantees the YOLOv8 val images are exactly Baseline A's 242 held-out val images.

In [ ]:
import json
import torch
from torch.utils.data import Dataset, random_split

class MalariaDataset(Dataset):
    def __init__(self, json_path, image_dir):
        self.image_dir = Path(image_dir)
        self._records = self._parse(Path(json_path))

    def _parse(self, json_path):
        with open(json_path) as f:
            raw = json.load(f)
        records = []
        for item in raw:
            img_name = Path(item['image']['pathname']).name
            boxes, labels = [], []
            for obj in item.get('objects', []):
                label = obj['category']
                if label in SKIP_LABELS or label not in LABEL_TO_INT:
                    continue
                bb = obj['bounding_box']
                x_min, y_min = float(bb['minimum']['c']), float(bb['minimum']['r'])
                x_max, y_max = float(bb['maximum']['c']), float(bb['maximum']['r'])
                if x_max <= x_min or y_max <= y_min:
                    continue
                boxes.append([x_min, y_min, x_max, y_max])
                labels.append(LABEL_TO_INT[label])
            if boxes:
                records.append({'img_name': img_name, 'boxes': boxes, 'labels': labels})
        return records

    def __len__(self):
        return len(self._records)

    def __getitem__(self, idx):
        return self._records[idx]

full_ds = MalariaDataset(TRAIN_JSON, IMG_DIR)
n_val   = int(len(full_ds) * 0.2)
n_train = len(full_ds) - n_val
gen     = torch.Generator().manual_seed(42)
train_sub, val_sub = random_split(full_ds, [n_train, n_val], generator=gen)

print(f'Total: {len(full_ds)}  ->  train {len(train_sub)} / val {len(val_sub)}')
assert len(train_sub) == 966 and len(val_sub) == 242, 'split does not match Baseline A -- check dataset version'

## 4. Convert to YOLO format

In [ ]:
import shutil
from PIL import Image

def yolo_class_idx(label_idx):
    return label_idx - 1

def write_split(records, img_dir, out_dir, split_name):
    img_out = out_dir / 'images' / split_name
    lbl_out = out_dir / 'labels' / split_name
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    written = 0
    for rec in records:
        img_path = Path(img_dir) / rec['img_name']
        if not img_path.exists():
            continue
        with Image.open(img_path) as im:
            w, h = im.size
        lines = []
        for (x_min, y_min, x_max, y_max), label_idx in zip(rec['boxes'], rec['labels']):
            cx = ((x_min + x_max) / 2) / w
            cy = ((y_min + y_max) / 2) / h
            bw = (x_max - x_min) / w
            bh = (y_max - y_min) / h
            lines.append(f'{yolo_class_idx(label_idx)} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
        shutil.copy(img_path, img_out / img_path.name)
        (lbl_out / (img_path.stem + '.txt')).write_text('\n'.join(lines))
        written += 1
    print(f'  {split_name}: {written}/{len(records)} images -> {img_out}')

train_records = [full_ds._records[i] for i in train_sub.indices]
val_records   = [full_ds._records[i] for i in val_sub.indices]

write_split(train_records, IMG_DIR, OUT_DIR, 'train')
write_split(val_records, IMG_DIR, OUT_DIR, 'val')

if TEST_JSON.exists():
    test_ds = MalariaDataset(TEST_JSON, IMG_DIR)
    write_split(test_ds._records, IMG_DIR, OUT_DIR, 'test')  # reference only (Stage-1 holdout in the paper)

yaml_text = (
    f"path: {OUT_DIR.resolve()}\n"
    f"train: images/train\n"
    f"val: images/val\n"
    f"test: images/test\n\n"
    f"names:\n" + '\n'.join(f'  {i}: {n}' for i, n in enumerate(FOREGROUND_NAMES)) + '\n'
)
(OUT_DIR / 'data.yaml').write_text(yaml_text)
print('\ndata.yaml:\n', yaml_text)

## 5. Train YOLOv8
80 epochs / imgsz 1024 to match Baseline A's protocol. Lower `batch` if you hit a CUDA OOM (Kaggle T4 has 16GB — batch=16 should fit `yolov8s` at imgsz 1024; drop to 8 if needed).

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')
model.train(
    data=str(OUT_DIR / 'data.yaml'),
    epochs=80,
    imgsz=1024,
    batch=16,
    seed=42,
    project='/kaggle/working',
    name='checkpoints-kaggle-80epoch',
    exist_ok=True,
    deterministic=True,
)

## 6. Evaluate on the val split (mAP@0.5, same metric as Baseline A)

In [ ]:
best_weights = CKPT_DIR / 'weights' / 'best.pt'
eval_model = YOLO(str(best_weights))
results = eval_model.val(data=str(OUT_DIR / 'data.yaml'), imgsz=1024, split='val', iou=0.5)

map50 = float(results.box.map50)
per_class_ap50 = results.box.ap50

yolo_metrics = {
    'map_50': round(map50, 4),
    'per_class_ap': {FOREGROUND_NAMES[i]: round(float(ap), 4) for i, ap in enumerate(per_class_ap50)},
    'model': 'YOLOv8s',
}

with open(CKPT_DIR / 'metrics.json', 'w') as f:
    json.dump(yolo_metrics, f, indent=2)

print(json.dumps(yolo_metrics, indent=2))

## 7. Compare against Baseline A (Faster R-CNN)
Baseline A numbers below are hard-coded from `Phase2-BaselineA/checkpoints-kaggle-80epoch/metrics.json` (80-epoch Kaggle run, mAP@0.5 = 58.99%).

In [ ]:
frcnn_metrics = {
    'map_50': 0.5899,
    'per_class_ap': {
        'red blood cell': 0.9094,
        'trophozoite':    0.6722,
        'ring':           0.5717,
        'schizont':       0.2457,
        'gametocyte':     0.2595,
        'leukocyte':      0.8812,
    },
}

print('| Class | Faster R-CNN (Baseline A) | YOLOv8s |')
print('|---|---|---|')
for c in frcnn_metrics['per_class_ap']:
    a = frcnn_metrics['per_class_ap'][c]
    b = yolo_metrics['per_class_ap'].get(c, float('nan'))
    print(f'| {c} | {100*a:.2f}% | {100*b:.2f}% |')
print(f"| **mAP@0.5** | **{100*frcnn_metrics['map_50']:.2f}%** | **{100*yolo_metrics['map_50']:.2f}%** |")

## 8. Download the results
After the run finishes, download `/kaggle/working/checkpoints-kaggle-80epoch/` (weights + metrics.json + training curves) from the notebook's Output pane to bring back into the `Phase5-YOLO-Baseline/` folder in the project repo.